<a href="https://colab.research.google.com/github/demsaid400-cpu/DI-BOOTCAMP/blob/main/Week7/Day4/Exercises_XP/Evaluating_LLMs_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [1]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [2]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [8]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    results = []
    for batch in batch_generator(texts, batch_size):
        inputs = tokenizer(["summarize: " + t for t in batch], return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = model.generate(inputs.input_ids, max_new_tokens=max_new_tokens)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(decoded)

        del inputs, outputs
        torch.cuda.empty_cache()
        gc.collect()
    return results

RUN_T5 = True
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=4)
    train_df['t5_small_summary'] = train_summaries_t5
    display(train_df[['prompt_text', 'prompt_title', 't5_small_summary']].head())

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

,prompt_text,prompt_title,t5_small_summary
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest..."
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [12]:
from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip().lower() == r.strip().lower())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")

Exact-match accuracy: 0.0000



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [9]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)  # Joining with newlines for ROUGE-L

def compute_rouge_score(preds: List[str], refs: List[str]):
    clean_preds = [normalize_text(p) for p in preds]
    clean_refs = [normalize_text(r) for r in refs]
    return rouge.compute(predictions=clean_preds, references=clean_refs)

print("ROUGE sanity check:")
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print(compute_rouge_score(test_preds, test_refs))

ROUGE sanity check:
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.6666666666666666), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


### Part VI - Results & Observations (Exercise Analysis)

Based on our ROUGE implementation, here are the observations for the requested experiments:

1. **Exact Match vs. Empty Prediction**:
   - An exact match (e.g., "alpha beta" vs "alpha beta") results in a score of **1.0**.
   - An empty prediction ("") results in a score of **0.0**, as there is no overlap with the reference.

2. **N-gram Overlap (ROUGE-1 vs ROUGE-2)**:
   - **ROUGE-1** (unigrams) is higher when individual words match, even if the order is wrong.
   - **ROUGE-2** (bigrams) drops significantly if the specific sequence of two words is broken, making it a better measure of fluency.

3. **Symmetry**:
   - ROUGE scores are calculated using the F1-measure. If you swap the prediction and the reference, the F1-score remains the same, although the individual Precision and Recall values would swap.

4. **ROUGE-L**:
   - It successfully captures the longest common subsequence, which is why it is often the preferred metric for summarization compared to strict accuracy.

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    results = []
    for batch in batch_generator(texts, batch_size):
        # GPT-2 style: Text + TL;DR:
        prompts = [t[:1000] + "\n\nTL;DR:" for t in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
        outputs = model.generate(inputs.input_ids, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)

        for i, out in enumerate(outputs):
            decoded = tokenizer.decode(out[inputs.input_ids.shape[1]:], skip_special_tokens=True)
            results.append(decoded.strip())

        torch.cuda.empty_cache()
        gc.collect()
    return results

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    scores = []
    for _, row in df.iterrows():
        score = rouge.compute(predictions=[normalize_text(row[pred_col])],
                              references=[normalize_text(row[ref_col])])
        scores.append(score['rougeL'])
    return scores

RUN_COMPARE = True
if RUN_COMPARE and 'train_summaries_t5' in locals():
    train_df['gpt2_summary'] = summarize_with_gpt2(train_df['prompt_text'].tolist())
    train_df['t5_rougeL'] = compute_rouge_per_row(train_df, 't5_small_summary')
    train_df['gpt2_rougeL'] = compute_rouge_per_row(train_df, 'gpt2_summary')
    print("Comparison complete.")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[tra

Comparison complete.



### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [11]:
import pandas as pd

def compare_models(df):
    # Calculate mean ROUGE scores for all models
    metrics = {
        'T5-Small': df['t5_rougeL'].mean(),
        'GPT-2': df['gpt2_rougeL'].mean()
    }
    return pd.DataFrame.from_dict(metrics, orient='index', columns=['Avg ROUGE-L'])

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    cols = ['prompt_title'] + pred_cols
    return df[cols].head(5)

if 't5_rougeL' in train_df.columns:
    display(compare_models(train_df))
    display(compare_models_summaries(train_df, ['t5_small_summary', 'gpt2_summary']))

,Avg ROUGE-L
T5-Small,0.154839
GPT-2,0.119918


,prompt_title,t5_small_summary,gpt2_summary
0,Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,"The first time I saw the new ""The Walking Dead..."
1,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...,The Aqua Dots toys contain a chemical that can...
2,The company has become a huge name in communic...,,Qualcomm is a big company. It's a big company....
3,NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest...","The first time I saw the new ""Star Wars"" movie..."
4,Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...,Julia Vakulenko will face comeback queen Linds...


## Wrap-up

### Reflection
- **Metrics**: ROUGE-L felt most informative because it accounts for longest common subsequences, capturing the flow of the summary better than simple n-gram overlap.
- **Model Size and Type**: The encoder-decoder architecture (T5) is much more efficient for summarization than the causal GPT-2 model at small scales. GPT-2 often requires a larger model (like GPT-2 Large or XL) and more sophisticated prompting to match T5-Small's focus.
- **Accuracy Breakdown**: Accuracy is a poor metric for summarization because there are infinitely many ways to summarize a text correctly; punishing a model for not picking the exact words of the reference is counter-productive.
- **Extensions**: To improve this, one could use **BERTScore** for semantic similarity or **Human Evaluation** to score fluency and factual consistency, which ROUGE often misses.